# Realistic scenario: sKF (Gaussian) vs sKF-L (exact) under heavy-tailed noise

`01_gaussiano.ipynb` and `02_laplaciano.ipynb` verified two closed forms against Augusto's
numerical integral of equation 18, on 3 weights. Here those same two closed forms, copied
unchanged, run on the scenario of section 7 of the draft: an acoustic channel of $M = 128$
coefficients, correlated input, heavy-tailed measurement noise. No numerical integration
anywhere.

Metric: misalignment $10\log_{10}\left(\|w_t - h_o\|^2/\|h_o\|^2\right)$ in dB, averaged over
20 independent realisations.

## 1. Imports

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import lfilter
from scipy.special import gammaln, log_ndtr, logsumexp
from scipy.stats import gennorm

print("imports ready")

## 2. The scenario

* **$M = 128$ coefficients.** Impulse response: exponential decay times white noise,
  normalised to unit norm. An **approximation** of a room impulse response, not a simulated
  room.
* **Fixed room.** The response is generated once, from its own seed, and never changes: no
  $T_c$.
* **$N = 96000$ steps, $R = 20$ realisations.**
* **Sampling rate 8 kHz**, the rate of the draft's scenario. It enters nothing in the
  recursions, only the time axis: the run is $96000/8000 = 12$ s, the same span as Fig. 3a of the
  published paper, so the figure can sit next to it without converting anything.
* **Input:** AR(1), $x_t = -0.9\,x_{t-1} + u_t$, unit variance. Correlated, as in the draft.
* **Measurement noise:** generalized Gaussian, shape $\beta = 0.2$, as in the draft. A
  Laplacian was tried first and the two filters land within a few tenths of a dB of each
  other: its tail is not heavy enough to separate them.
* **SNR $= 5$ dB**, the value of Fig. 3 of the published paper, from the draft's definition
  $10\log_{10}\left(h_o^{\mathsf T}R_{xx}h_o/v_\eta\right)$, with the signal power in closed
  form, $R_{xx}[i,j] = (-0.9)^{|i-j|}$. **Nominal, not realised:** with $\beta = 0.2$ the
  noise power of a finite run swings widely around $v_\eta$.
* **What the filters assume:** the Gaussian one gets $v_\eta$, the Laplacian one gets
  $b_\eta = \sqrt{v_\eta/2}$, the scale with that same variance. Neither is told the true
  shape $\beta = 0.2$, so both are mismatched.
* **Free parameter:** $\varepsilon$, one per filter, and nothing else. See section 4.

In [ ]:
M = 128                     # filter length / length of the impulse response
N = 96000                   # steps per run (12 s at 8 kHz, the span of Fig. 3a of the paper)
R = 20                      # independent realisations
WARMUP = 500                # AR samples discarded so the input starts stationary

AR_A = -0.9                 # AR(1) coefficient of the input
SNR_DB = 5.0                # nominal signal-to-noise ratio, as in Fig. 3 of the paper
BETA = 0.2                  # shape of the generalized Gaussian measurement noise
DECAY = 20.0                # decay constant, in samples, of the impulse response
VAR_THETA_0 = 2.0           # initial prior variance on each weight, as in notebooks 01 and 02
FS = 8000                   # sampling rate [Hz], only used to put the time axis in seconds

# The impulse response, generated once from its own seed and then fixed.
rng_room = np.random.default_rng(12345)
ho = np.exp(-np.arange(M)/DECAY)*rng_room.standard_normal(M)
ho = ho/np.linalg.norm(ho)

# Signal power, in closed form: r[k] = AR_A^|k| for a unit-variance AR(1) input.
lags = np.abs(np.subtract.outer(np.arange(M), np.arange(M)))
Rxx = AR_A**lags
P_signal = ho @ Rxx @ ho

var_eta = P_signal/10**(SNR_DB/10)                 # noise variance that gives SNR_DB
b_eta = np.sqrt(var_eta/2)                         # Laplacian scale with that same variance

# Scale of the generalized Gaussian with variance var_eta:
# Var = scale^2 Gamma(3/beta)/Gamma(1/beta).
scale_gg = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))

print(f"signal power = {P_signal:.4f}")
print(f"var_eta      = {var_eta:.4e}   (nominal, at {SNR_DB:.0f} dB SNR)")
print(f"b_eta        = {b_eta:.4e}")
print(f"noise scale  = {scale_gg:.4e}   (generalized Gaussian, beta = {BETA})")

One realisation: the input, the noisy observation, and the noise itself, kept only so that
Figure 2 can show it.

In [ ]:
def generate_signals(seed):
    """One realisation: AR(1) input, its clean output through ho, and heavy-tailed noise."""
    rng = np.random.default_rng(seed)
    u = np.sqrt(1 - AR_A**2)*rng.standard_normal(N + WARMUP)   # driving noise, unit-variance x
    x = lfilter([1.0], [1.0, -AR_A], u)[WARMUP:]               # x_t = AR_A x_{t-1} + u_t
    y = np.convolve(ho, x)[:N]                                 # clean output, y_t = x_t' ho
    eta = gennorm.rvs(BETA, scale=scale_gg, size=N, random_state=rng)
    return x, y + eta, eta


x, d, eta = generate_signals(0)
print("x:", x.shape, " d:", d.shape)
print(f"noise, first realisation: std {eta.std():.4f}, "
      f"median |eta| {np.median(np.abs(eta)):.5f}, max |eta| {np.abs(eta).max():.4f}")

## 3. The two closed forms

Copied from the notebooks that verified them, not reimplemented. **One change** in both: the
regularisation `x_reg = np.sign(x)*(np.abs(x) + 1e-3)` is gone, since it only existed to
reproduce Augusto's grid and there is no grid here.

Same notation in both: $e_t = d_t - x_t^{\mathsf T}w_{t-1}$ is the prediction error,
$\tilde v_t$ the predicted variance, $\varepsilon$ the process-noise variance, $M$ the number
of weights.

### sKF (Gaussian), equation (32) of the published paper

The paper writes it as $w_t = w_{t-1} + \kappa_t\alpha_t e_t$, with $\kappa_t = \bar V_t x_t$
the preliminary Kalman gain and $\alpha_t = h_t/(1 + h_t s_t)$ its multiplier, eqs. (26)–(33).
Under the scalar-variance model $\bar V_t = \tilde v_t I$ and, in the Gaussian case,
$h_t = 1/v_\eta$, that collapses to:

$$\tilde v_t = v_{t-1} + \varepsilon$$

$$\alpha_t = \frac{\tilde v_t}{v_\eta + \tilde v_t\|x_t\|^2}$$

$$w_t = w_{t-1} + \alpha_t\,x_t\,e_t$$

$$v_t = \tilde v_t\left(1 - \frac{\alpha_t\|x_t\|^2}{M}\right)$$

The correction is **a gain times the error**. It is proportional to $e_t$, so it is unbounded:
an outlier ten times larger moves the weights ten times further.

In [ ]:
def shift(new_x_sample, x_window):
    L = len(x_window)
    new_x_window = np.zeros(L)
    new_x_window[0] = new_x_sample
    new_x_window[1:] = x_window[:-1]
    return new_x_window


def sKF_closed_form(N, x, d, w0, parameters):
    """Closed form from the published paper (eq. 32): the same filter, solved by hand."""
    epsilon = parameters["epsilon"]                    # eps, process noise variance
    var_eta = parameters["var_eta"]                    # v_eta, observation noise variance
    L = len(w0)                                        # number of weights
    w = w0.copy()                                      # w, current weights, start at w0
    v = parameters["var_theta_0"]                      # v, scalar prior variance on each weight
    x_t = np.zeros(L)                                  # x_t, window with the last L samples
    w_hist = np.zeros((N, L))                          # weights at every step
    var_hist = np.zeros((N,))                          # variance at every step
    e = np.zeros((N,))                                 # e_t, prediction error at every step

    for k in range(N):
        x_t = shift(x[k], x_t)                         # new sample in, window slides
        e[k] = d[k] - x_t @ w                          # e_t = d_t - x_t' w_{t-1}

        w_hist[k] = w.copy()                           # state BEFORE the update, as Augusto does
        var_hist[k] = v                                # same for the variance

        if k >= L:                                     # Augusto waits for a full window; same here
            v_pred = v + epsilon                       # v_tilde = v + eps
            power = x_t @ x_t                          # ||x_t||^2
            gain = v_pred/(var_eta + v_pred*power)     # alpha = v_tilde / (v_eta + v_tilde ||x||^2)
            w = w + gain*x_t*e[k]                      # w = w + alpha x_t e_t
            v = v_pred*(1 - gain*power/L)              # v = v_tilde (1 - alpha ||x||^2 / L)

    return {"h": w, "e": e, "w_hist": w_hist, "var_hist": var_hist}


print("Gaussian closed form ready")

### sKF-L (minorized), section 4.1 of the draft

The minorization replaces the Laplacian log-density by a quadratic touching it at $e_t$,
which amounts to replacing the noise variance by an effective one,
$v^{\mathrm{eff}}_\eta = 1/h_t = b_\eta|e_t|$. Every step of the Gaussian case then applies
verbatim under $v_\eta \leftarrow b_\eta|e_t|$:

$$w_t = w_{t-1} + \frac{\tilde v_t\,x_t}{b_\eta|e_t| + \tilde v_t\|x_t\|^2}\,e_t$$

$$v_t = \tilde v_t\left(1 - \frac{1}{M}
        \frac{\tilde v_t\|x_t\|^2}{b_\eta|e_t| + \tilde v_t\|x_t\|^2}\right)$$

Read against the Gaussian filter above, the only change is $v_\eta \to b_\eta|e_t|$ in the
denominator: an outlier inflates the effective noise variance, shrinks the gain and
suppresses the update. The correction is bounded, like the exact filter's, but it approaches
the saturated limit as $O(1/|e_t|)$ rather than exponentially, which is where the draft
expects the two to differ.

The implementation is Ramiro's, copied unchanged from the shared repository.

In [ ]:
# === RAMIRO: sKF-L minorized (draft eqs. 50 and 51) - copied verbatim, NOT edited ===
# source: bayes-adaptive-filters.ipynb in this repository, cell "Adaptive Filters".
# It is a copy and not an import because the function lives inside a notebook cell, and
# importing it would mean executing that whole notebook, framework and Monte Carlo included.
# If it ever moves to a .py module, this cell should become an import.
#
# Its interface differs from the two filters above: the initial variance is read from
# parameters["v_tilde_0"], and the weight history comes back under the key "h". The call
# site adapts to that; the function is left exactly as it is. See Observations.
def sKF_L_algorithm(N, x, d, h0, parameters):
    h = h0
    epsilon = parameters["epsilon"]
    b_eta = parameters["b_eta"]
    v_tilde_0 = parameters["v_tilde_0"]

    # normalize x
    # regularization = 1e-3
    # x_reg = np.sign(x) * (np.abs(x) + regularization)

    L = len(h)
    y = np.zeros((N,))
    e = np.zeros((N,))
    xtemp = np.zeros(L)
    v=v_tilde_0
    h_hist = np.zeros((N, L))
    v_hist = np.zeros((N, L))
    # d: salida del sistema con ruido
    # y: salida estimada
    for k in range(0,N):
        xtemp = shift(x[k], xtemp)
        y[k] = h @ xtemp
        e[k] = d[k] - y[k]
        h_hist[k] = h
        v_hist[k] = v

        if k >= L:
            # predict
            v += epsilon
            # update
            norm = xtemp @ xtemp
            s = b_eta * abs(e[k]) + v * norm
            h = h + xtemp * (v * e[k]/s) # not h+= because it would mutate h0
            v = v * (1 - (v * norm) / (L * s)) 

    return {'h': h_hist, 'y': y, 'e': e, 'v': v_hist}
print("minorized closed form ready (Ramiro's implementation)")

### sKF-L (exact), section 5 of the draft

Same prediction, $\tilde v_t = v_{t-1} + \varepsilon$. The Laplacian likelihood splits the
marginal into two branches, $\varsigma = \pm 1$, and the correction is what is left after both
are integrated. With $\phi$, $\Phi$ the standard normal density and cdf:

$$\kappa_\varsigma = \frac{\varsigma e_t - \tilde v_t\|x_t\|^2/b_\eta}{\sqrt{\tilde v_t}\,\|x_t\|}
\qquad
\pi_\varsigma = \mathrm{softmax}\!\left(-\varsigma e_t/b_\eta + \log\Phi(\kappa_\varsigma)\right)
\qquad h(\kappa) = \phi(\kappa)/\Phi(\kappa)$$

Four global scalars, the same for every weight $m$:

$$\Lambda = \pi_+ - \pi_- \qquad \Gamma = \pi_+h(\kappa_+) - \pi_-h(\kappa_-) \qquad
P = \pi_+h(\kappa_+) + \pi_-h(\kappa_-) \qquad Q = \pi_+\kappa_+h(\kappa_+) + \pi_-\kappa_-h(\kappa_-)$$

and the update, with $\gamma = \tilde v_t/b_\eta$ and $\lambda = \sqrt{\tilde v_t}/\|x_t\|$:

$$w_t = w_{t-1} + \left(\frac{\tilde v_t}{b_\eta}\Lambda - \frac{\sqrt{\tilde v_t}}{\|x_t\|}\Gamma\right)x_t$$

$$D_t = \gamma^2(1-\Lambda^2) - 2\gamma\lambda(P - \Lambda\Gamma) - \lambda^2(Q + \Gamma^2)
\qquad v_t = \tilde v_t + \frac{1}{M}D_t\|x_t\|^2$$

**Where the three filters differ is one line.** The Gaussian one corrects by $\alpha_t e_t$,
unbounded in $e_t$. The minorized one uses the same gain with $v_\eta \to b_\eta|e_t|$, which
saturates as $O(1/|e_t|)$. This one has $\Lambda \in [-1, 1]$, which saturates exponentially:
a single observation moves each weight by at most $\tilde v_t|x_{t,m}|/b_\eta$, however large
the outlier is. Everything else, prediction and variance bookkeeping, plays the same role in
all three.

($\kappa_\pm$ here is the mixture argument, not the Kalman gain $\kappa_t$ of eq. 27. Note
also that everything is computed in the log domain: $\Phi(\kappa)$ underflows and
$e^{2e_t/b_\eta}$ overflows.)

In [ ]:
SIGN = np.array([1.0, -1.0])                           # the two branches, varsigma = +1 and -1


def sKF_L_closed_form(N, x, d, w0, parameters):
    """Closed form of section 5 (sKF-L, exact): the same filter, solved by hand."""
    epsilon = parameters["epsilon"]                    # eps, process noise variance
    b_eta = parameters["b_eta"]                        # b_eta, Laplacian observation scale
    M = len(w0)                                        # number of weights
    w = w0.copy()                                      # w, current weights, start at w0
    v = parameters["var_theta_0"]                      # v, scalar prior variance on each weight
    x_t = np.zeros(M)                                  # x_t, window with the last M samples
    w_hist = np.zeros((N, M))                          # weights at every step
    var_hist = np.zeros((N,))                          # variance at every step
    e = np.zeros((N,))                                 # e_t, prediction error at every step

    for k in range(N):
        x_t = shift(x[k], x_t)                         # new sample in, window slides
        e[k] = d[k] - x_t @ w                          # e_t = d_t - x_t' w_{t-1}

        w_hist[k] = w.copy()                           # state BEFORE the update, as Augusto does
        var_hist[k] = v                                # same for the variance

        if k >= M:                                     # Augusto waits for a full window; same here
            v_tilde = v + epsilon                      # v_tilde = v + eps
            norm_x = np.sqrt(x_t @ x_t)                # ||x_t||
            power = norm_x**2                          # ||x_t||^2

            # The two mixture arguments. Global: no dependence on the weight index m.
            kappa = (SIGN*e[k] - v_tilde*power/b_eta)/(np.sqrt(v_tilde)*norm_x)

            # Mixture weights, in the log domain. Phi(kappa) underflows and e^{2 e / b_eta}
            # overflows, so neither factor may be formed on its own.
            log_Phi = log_ndtr(kappa)                  # log Phi(kappa), accurate in the left tail
            log_pi = -SIGN*e[k]/b_eta + log_Phi
            pi = np.exp(log_pi - logsumexp(log_pi))    # softmax, sums to 1

            # Inverse Mills ratio h = phi/Phi, also in the log domain.
            log_phi = -0.5*kappa**2 - 0.5*np.log(2*np.pi)
            h = np.exp(log_phi - log_Phi)

            Lambda = np.sum(SIGN*pi)                   # saturating prediction error, in [-1, 1]
            Gamma = np.sum(SIGN*pi*h)
            P = np.sum(pi*h)
            Q = np.sum(pi*kappa*h)

            gain = v_tilde*Lambda/b_eta - np.sqrt(v_tilde)*Gamma/norm_x
            w = w + gain*x_t                           # w = w + (v~ L/b_eta - sqrt(v~) G/||x||) x

            gamma = v_tilde/b_eta
            lam = np.sqrt(v_tilde)/norm_x
            D = (gamma**2*(1 - Lambda**2)
                 - 2*gamma*lam*(P - Lambda*Gamma)
                 - lam**2*(Q + Gamma**2))
            v = v_tilde + D*power/M                    # v = v~ + D ||x||^2 / M

    return {"h": w, "e": e, "w_hist": w_hist, "var_hist": var_hist}


print("Laplacian closed form ready")

## 4. The run

**The target is chosen first and the parameters after**, as in the paper, not the other way
round. The target is $\bar{\mathsf{m}}_\infty = -20$ dB, which is what the published paper
pairs with SNR $= 5$ dB and $\beta^* = 0.2$ in its Fig. 3. $\varepsilon$ is the only free
parameter of each filter, and the three values below were picked **by hand**, out of about six
values per filter, so that all three floors land on that target. The systematic grid search of
section 7 is pending.

$-25$ dB, the deepest of the three targets in the paper's Fig. 4, was tried first and
abandoned: at 5 dB SNR the Gaussian filter never gets there. Its best floor is about
$-21.8$ dB after 150000 steps, still improving but far too slowly to be worth running. The
paper itself only uses the $-25$ dB target for robust filters.

In [ ]:
EPSILON_GAUSS = 3.0e-8      # hand-tuned, see the note above
EPSILON_MIN = 6.0e-6        # hand-tuned, for a matching floor
EPSILON_L = 3.0e-6          # hand-tuned, for a matching floor

parameters_gauss = {"epsilon": EPSILON_GAUSS, "var_eta": var_eta, "var_theta_0": VAR_THETA_0}
parameters_min = {"epsilon": EPSILON_MIN, "b_eta": b_eta, "v_tilde_0": VAR_THETA_0}
parameters_L = {"epsilon": EPSILON_L, "b_eta": b_eta, "var_theta_0": VAR_THETA_0}

w0 = np.zeros(M)                                   # all three filters start from zero
energy_ho = ho @ ho                                # ||ho||^2, the denominator of misalignment

misalignment_gauss = np.zeros(N)                   # running sums over the realisations
misalignment_min = np.zeros(N)
misalignment_L = np.zeros(N)

start = time.time()
for realisation in range(R):
    x, d, eta = generate_signals(realisation)

    w_gauss = sKF_closed_form(N, x, d, w0, parameters_gauss)["w_hist"]
    w_min = sKF_L_algorithm(N, x, d, w0, parameters_min)["h"]   # Ramiro returns it under "h"
    w_L = sKF_L_closed_form(N, x, d, w0, parameters_L)["w_hist"]

    misalignment_gauss += ((w_gauss - ho)**2).sum(axis=1)/energy_ho
    misalignment_min += ((w_min - ho)**2).sum(axis=1)/energy_ho
    misalignment_L += ((w_L - ho)**2).sum(axis=1)/energy_ho

misalignment_gauss /= R                            # ensemble average
misalignment_min /= R
misalignment_L /= R

print(f"{R} realisations of {N} steps, done in {time.time() - start:.0f} s")

## 5. Figures

**Figure 1.** Two panels, split the way the paper splits its Fig. 3 and the draft its section 7:
the filter that assumes Gaussian noise on the left, the two that assume Laplacian noise on the
right. Same $y$ axis in both so the heights compare directly; **independent $x$ axes**, because
the point is the difference in time scale.

In [ ]:
db_gauss = 10*np.log10(misalignment_gauss)
db_min = 10*np.log10(misalignment_min)
db_L = 10*np.log10(misalignment_L)
t_axis = np.arange(N)/FS                           # steps to seconds, as in Fig. 3 of the paper

T_ROBUST = 2.0                                     # seconds shown in (b): they settle well before
y_lim = (min(db_gauss.min(), db_min.min(), db_L.min()) - 1, 1)

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(12, 4.5))

ax_a.plot(t_axis, db_gauss, color="C0",
          label=f"sKF (Gaussian), eps = {EPSILON_GAUSS:.1e}", linewidth=1.2)
ax_a.set_title("(a) conventional, beta = 2.0")
ax_a.set_xlim(0, N/FS)

ax_b.plot(t_axis, db_min, color="C1",
          label=f"sKF-L (minorized), eps = {EPSILON_MIN:.1e}", linewidth=1.2)
ax_b.plot(t_axis, db_L, color="C2",
          label=f"sKF-L (exact), eps = {EPSILON_L:.1e}", linewidth=1.2)
ax_b.set_title("(b) robust, beta = 1.0")
ax_b.set_xlim(0, T_ROBUST)

for ax in (ax_a, ax_b):
    ax.set_xlabel("$t$ [sec]")
    ax.set_ylabel("misalignment [dB]")
    ax.set_ylim(*y_lim)
    ax.legend()
    ax.grid(alpha=0.3)

fig.suptitle(f"Convergence in heavy-tailed noise (beta* = {BETA}), SNR = {SNR_DB:.0f} dB, "
             f"average of {R} realisations")
plt.tight_layout()
plt.show()

And the parameters behind it, in the spirit of Table 1 of the published paper: the free
parameter of each filter and the floor it reaches with it, so the run can be reproduced.

In [ ]:
tail = slice(3*N//4, N)                            # last quarter of the run

table = [("sKF (Gaussian)", EPSILON_GAUSS, misalignment_gauss),
         ("sKF-L (minorized)", EPSILON_MIN, misalignment_min),
         ("sKF-L (exact)", EPSILON_L, misalignment_L)]

print(f"{'filter':<20}{'epsilon':>12}{'floor [dB]':>14}")
for name, epsilon, misalignment in table:
    print(f"{name:<20}{epsilon:>12.1e}{10*np.log10(misalignment[tail].mean()):>14.2f}")

**Figure 2.** One realisation of the measurement noise. Almost every sample sits invisibly
close to zero and a handful are enormous. The dashed line is the nominal $\sqrt{v_\eta}$, the
level both filters were told to expect.

In [ ]:
x, d, eta = generate_signals(0)

plt.figure(figsize=(10, 4))
plt.plot(eta, linewidth=0.6)
plt.axhline(np.sqrt(var_eta), color="k", linestyle="--", linewidth=0.8,
            label=r"nominal $\pm\sqrt{v_\eta}$")
plt.axhline(-np.sqrt(var_eta), color="k", linestyle="--", linewidth=0.8)
plt.xlabel("time step $t$")
plt.ylabel(r"$\eta_t$")
plt.title(f"Measurement noise, one realisation (generalized Gaussian, beta = {BETA})")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

median_eta = np.median(np.abs(eta))
print(f"median |eta| = {median_eta:.5f}")
print(f"nominal std  = {np.sqrt(var_eta):.5f}   ({np.sqrt(var_eta)/median_eta:.0f}x the median)")
print(f"max |eta|    = {np.abs(eta).max():.4f}   ({np.abs(eta).max()/median_eta:.0f}x the median)")

## 6. The numbers

Floor: the average over the last quarter of the run. Convergence: the first step within 3 dB
of that floor.

In [ ]:
tail = slice(3*N//4, N)


def steady_state(misalignment):
    """Floor in dB, and the first step within 3 dB of it."""
    floor = 10*np.log10(misalignment[tail].mean())
    db = 10*np.log10(misalignment)
    reached = int(np.argmax(db < floor + 3))
    return floor, reached


floor_gauss, reached_gauss = steady_state(misalignment_gauss)
floor_min, reached_min = steady_state(misalignment_min)
floor_L, reached_L = steady_state(misalignment_L)

print(f"sKF (Gaussian),    eps = {EPSILON_GAUSS:.1e}")
print(f"   steady-state misalignment : {floor_gauss:+.2f} dB")
print(f"   steps to floor + 3 dB     : {reached_gauss}")
print(f"sKF-L (minorized), eps = {EPSILON_MIN:.1e}")
print(f"   steady-state misalignment : {floor_min:+.2f} dB")
print(f"   steps to floor + 3 dB     : {reached_min}")
print(f"sKF-L (exact),     eps = {EPSILON_L:.1e}")
print(f"   steady-state misalignment : {floor_L:+.2f} dB")
print(f"   steps to floor + 3 dB     : {reached_L}")
print()
print(f"spread of the three floors      : {max(floor_gauss, floor_min, floor_L) - min(floor_gauss, floor_min, floor_L):.2f} dB")
print(f"exact vs Gaussian, speed-up     : {reached_gauss/reached_L:.2f}x")
print(f"exact vs minorized, speed-up    : {reached_min/reached_L:.2f}x")
print(f"exact vs minorized, floor gap   : {floor_L - floor_min:+.2f} dB")

## 7. Conclusion

**At the same target, the Laplacian filters get there nine times faster.** Floors of $-20.52$,
$-20.83$ and $-20.30$ dB, within $0.54$ dB of each other, reached in $6.15$ s by the Gaussian
filter against $0.54$ s and $0.68$ s by the minorized and the exact one. That ratio is what
Figure 1 is for: panel (a) needs 12 s of signal to show what panel (b) shows in 2. The
published paper reports the same thing under the same conditions, a tenfold difference between
the time scales of its Figs. 3a and 3b at SNR $= 5$ dB with $\beta^* = 0.2$. Nothing diverged:
the Gaussian filter was slowed, not broken.

**Exact against minorized: still very close.** $0.54$ dB of floor apart, the minorized one
reaching its floor in 4297 steps against 5461. It is marginally ahead on both counts, by
margins of the order of the hand-tuning granularity, so this run does not separate the two.
Expected outcome (iv) of the draft puts their difference at *intermediate* $|e_t|$, the two
agreeing in the saturated limit, and this scenario barely visits that middle range: with
$\beta = 0.2$ the noise is either negligible or enormous, so almost every step falls either
well inside the quadratic region, where the minorization is tight, or well into saturation,
where both filters agree by construction. This is not evidence against (iv); it is a scenario
that cannot test it.

Not shown here, from the tuning runs: at the *same* $\varepsilon = 3\times10^{-8}$ the exact
filter settles around $-35$ dB against $-20$ dB for the Gaussian one. Equalising the floor
trades that 15 dB away for a comparison of speed, which is what section 7 wants once there is
a $T_c$ to recover from.

## 8. Limitations

* **Room fixed.** No $T_c$, no switch from $h^{(1)}$ to $h^{(2)}$.
* **Recovery speed not measured.** The main criterion of section 7 of the draft is how fast a
  filter recovers after $T_c$; with a fixed room, that criterion is not exercised at all.
* **No equal-misalignment protocol.** Two $\varepsilon$ picked by hand, floors matching to
  about 0.8 dB, at a single target. No grid search, and no sweep over the target, which is
  where the draft expects the gap to widen.
* **Impulse response approximated.** Exponential decay times white noise, not the simulated
  $(5, 10, 6)$ m room at 8 kHz with $T_{60} = 200$ ms.
* **20 realisations, not 100.** With $\beta = 0.2$ the average is driven by rare samples, so
  the floors are worth a few tenths of a dB, no better.
* **Three filters only.** No fKF, no SG, no LF, the fully Laplacian prior of section 6.
* **One operating point.** 20 dB and $\beta = 0.2$, with no sweep over either, so nothing here
  locates the crossover, and, as the conclusion says, this shape cannot test exact against
  minorized.

## 9. Observations on the imported code

`sKF_L_algorithm` is Ramiro's, copied unchanged. Nothing below was fixed.

* Its two update lines match eqs. (50) and (51) of the draft term by term.
* `v_hist` is allocated `(N, L)` but holds one scalar variance repeated across the $L$
  columns. The other two filters here store a length-$N$ vector. It is 12 MB per call that
  nothing reads, not a wrong result.
* The weight history comes back under the key `h`, while `NLMS_algorithm`, in the same cell of
  his notebook, returns the *final* weights under that same key. Only bites if the two go into
  one loop.
* **Its floor is not monotonic in $\varepsilon$.** Below the optimum it gets worse again,
  because $v$ collapses and the filter freezes before converging. Two very different
  $\varepsilon$ can give the same floor, one of them a stall rather than a steady state. The
  $\varepsilon$ used here is on the upper branch, the same side as the other two filters.
  Worth knowing before the grid search of section 7.